# Notebook 10 — Final Evaluation & Reporting

**EvaliSense: AI-Powered Handwritten Examination Evaluation with ML-Based Grading Error Prediction**

---

## Objective

This notebook consolidates all experiments and generates a final evaluation report.

We will:
1. Summarize all pipeline component performance
2. Compile the model comparison table
3. Generate publication-ready figures
4. Compute overall system metrics
5. Discuss limitations and future work

**Why this matters:** The final notebook ties everything together — this is what you present in a thesis, report, or defense.

## Setup

In [ ]:
import os, sys, warnings, json, time
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import config

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 150

RESULTS_DIR = config.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

print("Setup complete ✓")

## 1. System Architecture Summary

```
╔═══════════════════════════════════════════════════════════════════════════════╗
║                        EvaliSense — System Architecture                     ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                             ║
║  [Answer Sheet]  →  [Preprocessing]  →  [Line Segmentation]                ║
║        ↓                                       ↓                            ║
║  Raw scan/photo       OpenCV pipeline     Projection-based                  ║
║                                                ↓                            ║
║                                      [TrOCR Recognition]                    ║
║                                      microsoft/trocr-base                   ║
║                                                ↓                            ║
║                                    [Rubric-Based Evaluation]                ║
║                                    Sentence-BERT + Keywords                 ║
║                                                ↓                            ║
║                                    [Feature Extraction (16-D)]              ║
║                                    OCR + Semantic + Score                   ║
║                                                ↓                            ║
║                                    [Risk Prediction (ML)]                   ║
║                                    RF/SVM/LR/GB classifier                  ║
║                                                ↓                            ║
║                                    [Examiner Dashboard]                     ║
║                                    Score + Risk Alert + Review              ║
║                                                                             ║
╚═══════════════════════════════════════════════════════════════════════════════╝
```

## 2. Component Performance Summary

In [ ]:
# Component summary table
components = [
    ('Preprocessing', 'OpenCV (grayscale + Otsu + morphology)', '< 0.5s', 'N/A'),
    ('Line Segmentation', 'Horizontal projection', '< 0.3s', 'N/A'),
    ('HTR Recognition', 'microsoft/trocr-base-handwritten (334M params)', '1-5s/line', 'CER, WER'),
    ('Answer Evaluation', 'all-MiniLM-L6-v2 (22M params)', '< 2s/answer', 'Accuracy vs expected'),
    ('Feature Extraction', '16-dimensional vector', '< 0.01s', 'N/A'),
    ('Risk Prediction', 'ML classifier (sklearn)', '< 0.01s', 'F1, ROC-AUC'),
]

print("\n" + "═" * 95)
print("  EVALISENSE — COMPONENT PERFORMANCE SUMMARY")
print("═" * 95)
print(f"{'Component':<22} {'Model/Method':<45} {'Latency':>12} {'Metrics':>12}")
print("─" * 95)
for name, model, latency, metrics in components:
    print(f"  {name:<20} {model:<45} {latency:>12} {metrics:>12}")
print("═" * 95)
print(f"  Total pipeline latency (CPU): ~30-90 seconds per answer sheet")
print(f"  Hardware: Intel Core Ultra 9 285H, 32GB RAM, CPU-only")

## 3. Model Comparison Table

In [ ]:
# Load data and run final comparison
syn_path = config.samples_dir / 'synthetic_dataset.jsonl'
records = []
with open(syn_path) as f:
    for line in f:
        records.append(json.loads(line.strip()))

feature_keys = sorted(records[0]['features'].keys())
X = np.array([[r['features'][k] for k in feature_keys] for r in records])
y = np.array([1 if r['is_grading_error'] else 0 for r in records])

models = {
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=10,
                                       class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=1.0, class_weight='balanced',
                    probability=True, random_state=RANDOM_STATE))
    ]),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=100, max_depth=5,
                                           learning_rate=0.1, random_state=RANDOM_STATE))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

comparison = []
print(f"\n{'Model':<22} {'Acc':>8} {'Prec':>8} {'Recall':>8} {'F1':>8} {'AUC':>8} {'Time':>8}")
print("═" * 75)

for name, pipeline in models.items():
    t0 = time.time()
    results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring)
    train_time = time.time() - t0
    
    row = {
        'Model': name,
        'Accuracy': results['test_accuracy'].mean(),
        'Precision': results['test_precision'].mean(),
        'Recall': results['test_recall'].mean(),
        'F1': results['test_f1'].mean(),
        'ROC-AUC': results['test_roc_auc'].mean(),
        'Time': train_time,
    }
    comparison.append(row)
    print(f"  {name:<20} {row['Accuracy']:>7.4f} {row['Precision']:>7.4f} {row['Recall']:>7.4f} "
          f"{row['F1']:>7.4f} {row['ROC-AUC']:>7.4f} {train_time:>6.1f}s")

print("═" * 75)

best = max(comparison, key=lambda r: r['F1'])
print(f"\n✓ Best model: {best['Model']} (F1 = {best['F1']:.4f}, AUC = {best['ROC-AUC']:.4f})")

In [ ]:
# Publication-ready comparison chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Comparison — Grading Error Prediction', fontsize=16, fontweight='bold')

model_names = [r['Model'] for r in comparison]
model_colors = ['#4A90D9', '#E8744F', '#50C878', '#9B59B6']

# Grouped bar chart
ax = axes[0]
metrics_show = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
x = np.arange(len(model_names))
width = 0.15

for i, metric in enumerate(metrics_show):
    vals = [r[metric] for r in comparison]
    ax.bar(x + i * width, vals, width, label=metric, alpha=0.85)

ax.set_xticks(x + width * 2)
ax.set_xticklabels([n.split()[0] for n in model_names], fontsize=9)
ax.set_ylabel('Score')
ax.set_title('All Metrics')
ax.legend(fontsize=8, ncol=5, loc='lower right')
ax.set_ylim(0, 1.05)

# F1 comparison (highlight best)
ax = axes[1]
f1_vals = [r['F1'] for r in comparison]
bar_colors = [c if r['Model'] != best['Model'] else '#2ECC71' for r, c in zip(comparison, model_colors)]
bars = ax.bar(model_names, f1_vals, color=bar_colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.4f}',
            ha='center', fontsize=10, fontweight='bold')

ax.set_ylabel('F1 Score')
ax.set_title(f'F1 Score (best: {best["Model"]})', fontsize=12)
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'final_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR / 'final_model_comparison.png'}")

## 4. Dataset Summary

In [ ]:
# Count all data assets
iam_meta = config.data_dir / 'external' / 'iam_lines' / 'metadata.jsonl'
iam_count = 0
if iam_meta.exists():
    with open(iam_meta) as f:
        iam_count = sum(1 for _ in f)

rubric_count = len(list(config.samples_dir.glob('rubric_*.json')))
n_features = len(feature_keys)

print("\n" + "═" * 60)
print("  DATASET SUMMARY")
print("═" * 60)
print(f"  IAM handwriting images:       {iam_count:>8,}")
print(f"  Sample rubrics:               {rubric_count:>8}")
print(f"  Synthetic training records:   {len(records):>8}")
print(f"  Feature dimensions:           {n_features:>8}")
print(f"  Grading error threshold:      {2.0:>8.1f} marks")
print(f"  Error rate:                   {np.mean(y):>7.1%}")
print("═" * 60)

## 5. Key Results Summary

### HTR Performance
- TrOCR achieves competitive CER on IAM handwriting benchmark
- CPU-executable in 1-5 seconds per line
- Confidence scores are well-calibrated and useful for risk prediction

### Evaluation Accuracy
- Sentence-BERT evaluation correctly discriminates good/partial/poor answers
- AI scores fall within expected ranges for validated test cases
- Keyword matching provides complementary signal to semantic similarity

### Risk Prediction
- Best model achieves strong F1 and ROC-AUC on grading error detection
- `confidence_score_disagreement` and `score_variance` are the most predictive features
- `class_weight='balanced'` is essential for the ~22% error rate imbalance

In [ ]:
# Generate summary table for report
print("\n" + "═" * 70)
print("  FINAL RESULTS — GRADING ERROR PREDICTION")
print("═" * 70)
print(f"  {'Metric':<30} {'Value':>12}")
print("  " + "─" * 45)
print(f"  {'Best Model':<30} {best['Model']:>12}")
print(f"  {'Accuracy':<30} {best['Accuracy']:>12.4f}")
print(f"  {'Precision':<30} {best['Precision']:>12.4f}")
print(f"  {'Recall':<30} {best['Recall']:>12.4f}")
print(f"  {'F1 Score':<30} {best['F1']:>12.4f}")
print(f"  {'ROC-AUC':<30} {best['ROC-AUC']:>12.4f}")
print(f"  {'Training Data':<30} {len(records):>12} records")
print(f"  {'Feature Count':<30} {n_features:>12}")
print(f"  {'Cross-Validation':<30} {'5-fold stratified':>12}")
print(f"  {'Class Weighting':<30} {'balanced':>12}")
print(f"  {'Device':<30} {'CPU':>12}")
print("═" * 70)

## 6. Limitations & Future Work

### Current Limitations
1. **Synthetic training data** — The risk model is trained on synthetic records. Real-world grading patterns may differ.
2. **CPU-only** — GPU acceleration would reduce HTR time from seconds to milliseconds.
3. **English only** — TrOCR is trained on English handwriting. Multilingual support requires additional models.
4. **Simple segmentation** — The projection-based line segmenter may fail on heavily overlapping text.

### Future Work
1. **Collect real grading data** — Partner with educational institutions to build a real training set with examiner disagreement labels.
2. **Fine-tune TrOCR** — Domain-specific fine-tuning on exam handwriting could significantly reduce CER.
3. **Active learning** — Use examiner corrections to continuously improve the risk model.
4. **Multi-question support** — Extend to handle multiple questions per answer sheet with automatic question detection.
5. **Explanation generation** — Generate human-readable explanations for why a particular answer received its score.

## Conclusion

EvaliSense demonstrates a complete, CPU-executable pipeline for AI-assisted examination grading. The system:
- Digitizes handwritten answers using TrOCR
- Evaluates answers against structured rubrics with semantic similarity
- Predicts grading errors using ML classification
- Assists — not replaces — the human examiner

The approach is practical, interpretable, and deployable on standard hardware.